In [1]:
import ee

PROJECT_ID = "lst-gee"  # <-- replace with your real GCP project ID

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()  # opens an OAuth flow, click through and paste/allow
    ee.Initialize(project=PROJECT_ID)

print("EE initialized:", ee.String("ok").getInfo())

EE initialized: ok


In [5]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132

In [16]:
import os
import hashlib
import glob
import json
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
import mlflow
import rasterio
from rasterio.windows import Window
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import glob
import json
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchgeo.models import ViTSmall16_Weights
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import mlflow

In [13]:
!pip install torchgeo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.1/688.1 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 871.8/871.8 kB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
 #Region definitions

# Small, representative agricultural sub-regions (~12-18 km across), NOT full states.
# Chosen for known cropping activity + reasonable cloud-free imagery availability.
#
# Punjab: Ludhiana district belt — wheat/paddy rotation (Rabi/Kharif)
# Maharashtra: Nashik belt — grape/onion/vegetable mixed agriculture
# Karnataka: Raichur belt — rice/cotton, Krishna-Tungabhadra command area
#
# NOTE: these coordinates are approximate agricultural zones for prototyping.
# If you have a specific district/field boundary in mind, swap the coordinates below.

REGIONS = {
    "punjab_ludhiana": ee.Geometry.Rectangle([75.70, 30.80, 75.90, 30.98]),
    "maharashtra_nashik": ee.Geometry.Rectangle([73.70, 19.90, 73.90, 20.08]),
    "karnataka_raichur": ee.Geometry.Rectangle([76.30, 16.10, 76.50, 16.28]),
}

# Print area of each AOI as a sanity check
for name, geom in REGIONS.items():
    area_km2 = geom.area().divide(1e6).getInfo()
    print(f"{name}: {area_km2:.1f} km^2")

punjab_ludhiana: 382.0 km^2
maharashtra_nashik: 418.3 km^2
karnataka_raichur: 427.5 km^2


In [3]:
# Seasonal date windows (captures phenology across the crop calendar)

# India's crop calendar roughly: Kharif (Jun-Oct), Rabi (Nov-Mar), Zaid/summer (Mar-Jun)
# 4 windows per region chosen to span sowing -> peak growth -> harvest -> fallow.

SEASONS = {
    "kharif_peak":        ("2023-09-01", "2023-09-30"),
    "kharif_harvest_rabi_sow": ("2023-11-15", "2023-12-15"),
    "rabi_peak":          ("2024-01-15", "2024-02-15"),
    "rabi_harvest_fallow": ("2024-04-01", "2024-04-30"),
}

In [4]:
# Cloud masking (s2cloudless method)

CLOUD_FILTER = 60        # max scene-level cloud % to even consider
CLD_PRB_THRESH = 40      # cloud probability threshold (0-100)
NIR_DRK_THRESH = 0.15    # cloud shadow detection threshold on NIR
CLD_PRJ_DIST = 2         # km to project cloud shadows
BUFFER = 100             # m buffer around detected clouds

def get_s2_sr_cld_col(aoi, start_date, end_date):
    """Join Sentinel-2 SR (L2A) with the s2cloudless probability collection."""
    s2_sr_col = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", CLOUD_FILTER))
    )
    s2_cloudless_col = (
        ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
    )
    return ee.ImageCollection(
        ee.Join.saveFirst("s2cloudless").apply(
            primary=s2_sr_col,
            secondary=s2_cloudless_col,
            condition=ee.Filter.equals(leftField="system:index", rightField="system:index"),
        )
    )

def add_cloud_shadow_mask(img):
    cld_prb = ee.Image(img.get("s2cloudless")).select("probability")
    is_cloud = cld_prb.gt(CLD_PRB_THRESH).rename("clouds")

    # Cloud shadow: dark NIR pixels projected from cloud edges
    dark_pixels = img.select("B8").lt(NIR_DRK_THRESH * 1e4).rename("dark_pixels")
    shadow_azimuth = ee.Number(90).subtract(ee.Number(img.get("MEAN_SOLAR_AZIMUTH_ANGLE")))
    cld_proj = (
        is_cloud.directionalDistanceTransform(shadow_azimuth, CLD_PRJ_DIST * 10)
        .reproject(crs=img.select("B2").projection(), scale=100)
        .select("distance")
        .mask()
        .rename("cloud_transform")
    )
    shadows = cld_proj.multiply(dark_pixels).rename("shadows")

    is_cld_shdw = is_cloud.add(shadows).gt(0)
    is_cld_shdw = (
        is_cld_shdw.focal_min(2).focal_max(BUFFER * 2 / 20)
        .reproject(crs=img.select("B2").projection(), scale=20)
        .rename("cloudmask")
    )
    return img.addBands(is_cld_shdw)

def apply_cloud_mask(img):
    not_cld_shdw = img.select("cloudmask").eq(0)
    return img.select(
        ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
    ).updateMask(not_cld_shdw)

def build_masked_composite(aoi, start_date, end_date):
    """Cloud-masked median composite for one region/season."""
    col = get_s2_sr_cld_col(aoi, start_date, end_date)
    masked = col.map(add_cloud_shadow_mask).map(apply_cloud_mask)
    return masked.median().clip(aoi)

In [5]:
EXPORT_BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
SCALE = 10  # meters/pixel (20m bands will be resampled to 10m on export)
BYTES_PER_PIXEL = 2  # Sentinel-2 SR bands are uint16

def dry_run():
    print(f"{'region':<22}{'season':<28}{'imgs':<6}{'est_MB':>10}")
    total_mb = 0
    plan = []
    for region_name, geom in REGIONS.items():
        area_m2 = geom.area().getInfo()
        n_pixels = area_m2 / (SCALE ** 2)
        est_bytes = n_pixels * len(EXPORT_BANDS) * BYTES_PER_PIXEL
        est_mb = est_bytes / 1e6

        for season_name, (start, end) in SEASONS.items():
            col = get_s2_sr_cld_col(geom, start, end)
            n_imgs = col.size().getInfo()
            print(f"{region_name:<22}{season_name:<28}{n_imgs:<6}{est_mb:>9.1f}")
            total_mb += est_mb
            plan.append(
                {
                    "region": region_name,
                    "season": season_name,
                    "start": start,
                    "end": end,
                    "n_source_images": n_imgs,
                    "est_mb": round(est_mb, 1),
                }
            )
    print(f"\nEstimated total export size: {total_mb:.1f} MB ({total_mb/1024:.2f} GB)")
    if total_mb / 1024 > 1.5:
        print("WARNING: exceeds 1.5 GB target — consider shrinking AOI or dropping a season.")
    return plan

plan = dry_run()


region                season                      imgs      est_MB
punjab_ludhiana       kharif_peak                 3          76.4
punjab_ludhiana       kharif_harvest_rabi_sow     6          76.4
punjab_ludhiana       rabi_peak                   4          76.4
punjab_ludhiana       rabi_harvest_fallow         5          76.4
maharashtra_nashik    kharif_peak                 3          83.7
maharashtra_nashik    kharif_harvest_rabi_sow     13         83.7
maharashtra_nashik    rabi_peak                   10         83.7
maharashtra_nashik    rabi_harvest_fallow         11         83.7
karnataka_raichur     kharif_peak                 3          85.5
karnataka_raichur     kharif_harvest_rabi_sow     9          85.5
karnataka_raichur     rabi_peak                   12         85.5
karnataka_raichur     rabi_harvest_fallow         8          85.5

Estimated total export size: 982.2 MB (0.96 GB)


In [7]:
#EXPORT
RUN_EXPORT = True  # <-- flip to True only when you're ready to trigger real exports
DRIVE_FOLDER = "crop_classification_exports"

if RUN_EXPORT:
    tasks = []
    for region_name, geom in REGIONS.items():
        for season_name, (start, end) in SEASONS.items():
            composite = build_masked_composite(geom, start, end).select(EXPORT_BANDS)
            description = f"{region_name}_{season_name}"
            task = ee.batch.Export.image.toDrive(
                image=composite.toUint16(),
                description=description,
                folder=DRIVE_FOLDER,
                fileNamePrefix=description,
                region=geom,
                scale=SCALE,
                crs="EPSG:4326",
                maxPixels=1e9,
                fileFormat="GeoTIFF",
            )
            task.start()
            tasks.append(task)
            print(f"Started export task: {description}")

    print(f"\n{len(tasks)} export tasks submitted. Check progress at:")
    print("https://code.earthengine.google.com/tasks")
else:
    print("RUN_EXPORT is False — no exports triggered. Review the dry run, then set RUN_EXPORT=True.")

Started export task: punjab_ludhiana_kharif_peak
Started export task: punjab_ludhiana_kharif_harvest_rabi_sow
Started export task: punjab_ludhiana_rabi_peak
Started export task: punjab_ludhiana_rabi_harvest_fallow
Started export task: maharashtra_nashik_kharif_peak
Started export task: maharashtra_nashik_kharif_harvest_rabi_sow
Started export task: maharashtra_nashik_rabi_peak
Started export task: maharashtra_nashik_rabi_harvest_fallow
Started export task: karnataka_raichur_kharif_peak
Started export task: karnataka_raichur_kharif_harvest_rabi_sow
Started export task: karnataka_raichur_rabi_peak
Started export task: karnataka_raichur_rabi_harvest_fallow

12 export tasks submitted. Check progress at:
https://code.earthengine.google.com/tasks


In [17]:
# LABELS (ESA WorldCover cropland mask, GEE-native)
REGIONS = {
    "punjab_ludhiana": ee.Geometry.Rectangle([75.70, 30.80, 75.90, 30.98]),
    "maharashtra_nashik": ee.Geometry.Rectangle([73.70, 19.90, 73.90, 20.08]),
    "karnataka_raichur": ee.Geometry.Rectangle([76.30, 16.10, 76.50, 16.28]),
}

CROPLAND_CLASS_CODE = 40  # ESA WorldCover v200 "Cropland"
DRIVE_FOLDER = "crop_classification_exports"
RUN_EXPORT = True

worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")

def build_binary_mask(aoi):
    """1 = cropland, 0 = everything else, clipped to AOI."""
    return worldcover.eq(CROPLAND_CLASS_CODE).rename("cropland").clip(aoi).toUint8()

def dry_run():
    print(f"{'region':<22}{'cropland_%':>12}")
    for name, geom in REGIONS.items():
        mask = build_binary_mask(geom)
        stats = mask.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geom,
            scale=10,
            maxPixels=1e9,
        ).getInfo()
        pct = stats.get("cropland", 0) * 100
        print(f"{name:<22}{pct:>11.1f}%")
        if pct < 5:
            print(f"  WARNING: {name} has very low cropland coverage — "
                  f"consider relocating the AOI.")

dry_run()



region                  cropland_%
punjab_ludhiana              52.1%
maharashtra_nashik           45.4%
karnataka_raichur            59.9%


In [18]:
if RUN_EXPORT:
    tasks = []
    for name, geom in REGIONS.items():
        mask = build_binary_mask(geom)
        description = f"{name}_worldcover_mask"
        task = ee.batch.Export.image.toDrive(
            image=mask,
            description=description,
            folder=DRIVE_FOLDER,
            fileNamePrefix=description,
            region=geom,
            scale=10,
            crs="EPSG:4326",
            maxPixels=1e9,
            fileFormat="GeoTIFF",
        )
        task.start()
        tasks.append(task)
        print(f"Started export: {description}")
    print(f"\n{len(tasks)} mask export tasks submitted. Check "
          f"https://code.earthengine.google.com/tasks")
else:
    print("RUN_EXPORT is False — no exports triggered. Review cropland % above, "
          "then set RUN_EXPORT=True.")

Started export: punjab_ludhiana_worldcover_mask
Started export: maharashtra_nashik_worldcover_mask
Started export: karnataka_raichur_worldcover_mask

3 mask export tasks submitted. Check https://code.earthengine.google.com/tasks


In [7]:
# STAGE 3  PREPROCESSING
DRIVE_DIR = "/content/drive/MyDrive/crop_classification_exports"  # adjust if needed
OUTPUT_DIR = "/content/patches"  # local Colab disk — fast for training
PATCH_SIZE = 64
GRID_BLOCK_SIZE = 4  # patches per grid block edge, i.e. block = 4x4 patches
CROPLAND_FRACTION_THRESHOLD = 0.5  # patch labeled "cropland" if >=50% of pixels are cropland

REGIONS = ["punjab_ludhiana", "maharashtra_nashik", "karnataka_raichur"]
SEASONS = ["kharif_peak", "kharif_harvest_rabi_sow", "rabi_peak", "rabi_harvest_fallow"]

S2_BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
# Band index positions within the exported GeoTIFF (order matches Stage 1 EXPORT_BANDS)
BLUE, GREEN, RED, RE1, RE2, RE3, NIR, RE4, SWIR1, SWIR2 = range(10)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [8]:
# Spectral indices
def compute_indices(bands, eps=1e-6):
    """
    bands: float32 array [C, H, W] in Sentinel-2 SR reflectance*10000 units,
           channel order matching S2_BANDS above.
    Returns NDVI, NDWI, EVI stacked as [3, H, W], each roughly in [-1, 1]
    (EVI is not strictly bounded but typically falls in that range for
    vegetated land).
    """
    b = bands.astype(np.float32) / 10000.0  # scale to reflectance [0, ~1]
    red, nir, green, blue = b[RED], b[NIR], b[GREEN], b[BLUE]

    ndvi = (nir - red) / (nir + red + eps)
    ndwi = (green - nir) / (green + nir + eps)  # McFeeters NDWI (water/moisture)
    evi = 2.5 * (nir - red) / (nir + 6 * red - 7.5 * blue + 1 + eps)

    return np.stack([ndvi, ndwi, evi], axis=0)

In [9]:
# Patch extraction
# ------------------------------------------------------------
def block_id_for_pixel(row, col, patch_size=PATCH_SIZE, block_size=GRID_BLOCK_SIZE):
    patch_row, patch_col = row // patch_size, col // patch_size
    return (patch_row // block_size, patch_col // block_size)


def assign_split(region, block_row, block_col, train_frac=0.70, val_frac=0.15):
    """Deterministic hash-based split assignment per (region, block) —
    reproducible across runs without storing a lookup table."""
    key = f"{region}_{block_row}_{block_col}".encode()
    h = int(hashlib.md5(key).hexdigest(), 16) % 1000 / 1000.0
    if h < train_frac:
        return "train"
    elif h < train_frac + val_frac:
        return "val"
    else:
        return "test"


def extract_patches_for_region_season(region, season):
    img_path = os.path.join(DRIVE_DIR, f"{region}_{season}.tif")
    mask_path = os.path.join(DRIVE_DIR, f"{region}_worldcover_mask.tif")

    if not os.path.exists(img_path):
        print(f"  SKIP (missing): {img_path}")
        return []
    if not os.path.exists(mask_path):
        print(f"  SKIP (missing mask): {mask_path}")
        return []

    records = []
    with rasterio.open(img_path) as img_src, rasterio.open(mask_path) as mask_src:
        h, w = img_src.height, img_src.width
        n_rows, n_cols = h // PATCH_SIZE, w // PATCH_SIZE

        for pr in range(n_rows):
            for pc in range(n_cols):
                row_off, col_off = pr * PATCH_SIZE, pc * PATCH_SIZE
                window = Window(col_off, row_off, PATCH_SIZE, PATCH_SIZE)

                img_patch = img_src.read(window=window).astype(np.float32)  # [10, P, P]
                if img_patch.shape[1:] != (PATCH_SIZE, PATCH_SIZE):
                    continue  # partial edge patch, skip
                if np.all(img_patch == 0):
                    continue  # fully masked/no-data patch (cloud-masked composite gap)

                # Mask is exported at the same 10m/EPSG:4326 grid as the imagery,
                # so pixel windows align directly.
                mask_patch = mask_src.read(1, window=window)
                if mask_patch.shape != (PATCH_SIZE, PATCH_SIZE):
                    continue
                cropland_frac = float(mask_patch.mean())
                label = int(cropland_frac >= CROPLAND_FRACTION_THRESHOLD)

                indices = compute_indices(img_patch)
                stacked = np.concatenate([img_patch, indices], axis=0)  # [13, P, P]

                block_row, block_col = block_id_for_pixel(row_off, col_off)
                split = assign_split(region, block_row, block_col)

                out_name = f"{region}_{season}_r{pr}_c{pc}.npz"
                out_path = os.path.join(OUTPUT_DIR, split, out_name)
                os.makedirs(os.path.dirname(out_path), exist_ok=True)
                np.savez_compressed(
                    out_path,
                    features=stacked.astype(np.float32),
                    pixel_mask=mask_patch.astype(np.uint8),
                    label=label,
                    cropland_frac=cropland_frac,
                    region=region,
                    season=season,
                )
                records.append({"path": out_path, "split": split, "label": label,
                                 "region": region, "season": season})
    return records


def run_preprocessing():
    all_records = []
    for region in REGIONS:
        for season in SEASONS:
            print(f"Processing {region} / {season} ...")
            recs = extract_patches_for_region_season(region, season)
            print(f"  -> {len(recs)} patches")
            all_records.extend(recs)

    counts = {}
    for r in all_records:
        key = (r["split"], r["label"])
        counts[key] = counts.get(key, 0) + 1

    print("\n=== Patch summary ===")
    print(f"Total patches: {len(all_records)}")
    for split in ["train", "val", "test"]:
        n_crop = counts.get((split, 1), 0)
        n_noncrop = counts.get((split, 0), 0)
        total = n_crop + n_noncrop
        print(f"  {split:<6}: {total:>5} patches  "
              f"(cropland={n_crop}, non-cropland={n_noncrop})")
        if total == 0:
            print(f"    WARNING: no patches in '{split}' split — "
                  f"check that Stage 1/2 exports downloaded correctly.")
    return all_records


if __name__ == "__main__":
    records = run_preprocessing()


Processing punjab_ludhiana / kharif_peak ...
  -> 1054 patches
Processing punjab_ludhiana / kharif_harvest_rabi_sow ...
  SKIP (missing): /content/drive/MyDrive/crop_classification_exports/punjab_ludhiana_kharif_harvest_rabi_sow.tif
  -> 0 patches
Processing punjab_ludhiana / rabi_peak ...
  -> 1054 patches
Processing punjab_ludhiana / rabi_harvest_fallow ...
  -> 1054 patches
Processing maharashtra_nashik / kharif_peak ...
  -> 714 patches
Processing maharashtra_nashik / kharif_harvest_rabi_sow ...
  -> 1054 patches
Processing maharashtra_nashik / rabi_peak ...
  -> 1054 patches
Processing maharashtra_nashik / rabi_harvest_fallow ...
  -> 1054 patches
Processing karnataka_raichur / kharif_peak ...
  -> 992 patches
Processing karnataka_raichur / kharif_harvest_rabi_sow ...
  -> 1054 patches
Processing karnataka_raichur / rabi_peak ...
  -> 1054 patches
Processing karnataka_raichur / rabi_harvest_fallow ...
  -> 1054 patches

=== Patch summary ===
Total patches: 11192
  train :  7158 pa

In [10]:
#STAGE 4a — RANDOM FOREST BASELINE
PATCH_DIR = "/content/patches"
RESULTS_PATH = "/content/results.json"
CLASS_NAMES = ["non_cropland", "cropland"]


def load_split(split):
    paths = sorted(glob.glob(os.path.join(PATCH_DIR, split, "*.npz")))
    X, y = [], []
    for p in paths:
        d = np.load(p)
        feats = d["features"]  # [13, H, W]
        means = feats.mean(axis=(1, 2))
        stds = feats.std(axis=(1, 2))
        X.append(np.concatenate([means, stds]))
        y.append(int(d["label"]))
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64), paths


def append_results(entry):
    results = []
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH) as f:
            results = json.load(f)
    results.append(entry)
    with open(RESULTS_PATH, "w") as f:
        json.dump(results, f, indent=2)


def main():
    print("Loading patches...")
    X_train, y_train, _ = load_split("train")
    X_val, y_val, _ = load_split("val")
    X_test, y_test, _ = load_split("test")
    print(f"train={len(y_train)}  val={len(y_val)}  test={len(y_test)}")

    if len(y_train) == 0 or len(y_test) == 0:
        raise RuntimeError(
            "No train/test patches found — run Stage 3 preprocessing first "
            f"and confirm .npz files exist under {PATCH_DIR}."
        )

    mlflow.set_experiment("crop_classification")
    with mlflow.start_run(run_name="random_forest_baseline"):
        params = dict(n_estimators=300, max_depth=None, min_samples_leaf=2,
                       class_weight="balanced", random_state=42, n_jobs=-1)
        mlflow.log_params(params)

        clf = RandomForestClassifier(**params)
        clf.fit(X_train, y_train)

        val_pred = clf.predict(X_val) if len(y_val) else None
        test_pred = clf.predict(X_test)

        acc = accuracy_score(y_test, test_pred)
        f1_macro = f1_score(y_test, test_pred, average="macro")
        f1_per_class = f1_score(y_test, test_pred, average=None)
        cm = confusion_matrix(y_test, test_pred)

        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_f1_macro", f1_macro)
        for cname, f1c in zip(CLASS_NAMES, f1_per_class):
            mlflow.log_metric(f"test_f1_{cname}", f1c)
        if val_pred is not None and len(y_val):
            mlflow.log_metric("val_accuracy", accuracy_score(y_val, val_pred))

        print("\n=== Random Forest — Test Set ===")
        print(f"Accuracy: {acc:.4f}")
        print(f"Macro F1: {f1_macro:.4f}")
        print(classification_report(y_test, test_pred, target_names=CLASS_NAMES))
        print("Confusion matrix (rows=true, cols=pred):")
        print(cm)

        append_results({
            "model": "random_forest",
            "test_accuracy": float(acc),
            "test_f1_macro": float(f1_macro),
            "test_f1_per_class": {c: float(v) for c, v in zip(CLASS_NAMES, f1_per_class)},
            "confusion_matrix": cm.tolist(),
            "n_train": len(y_train), "n_val": len(y_val), "n_test": len(y_test),
        })

    return clf


if __name__ == "__main__":
    main()


Loading patches...
train=7158  val=1927  test=2107


2026/07/21 07:30:26 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/21 07:30:26 INFO mlflow.store.db.utils: Updating database tables
2026/07/21 07:30:29 INFO mlflow.tracking.fluent: Experiment with name 'crop_classification' does not exist. Creating a new experiment.



=== Random Forest — Test Set ===
Accuracy: 0.8671
Macro F1: 0.8669
              precision    recall  f1-score   support

non_cropland       0.87      0.86      0.86      1018
    cropland       0.87      0.88      0.87      1089

    accuracy                           0.87      2107
   macro avg       0.87      0.87      0.87      2107
weighted avg       0.87      0.87      0.87      2107

Confusion matrix (rows=true, cols=pred):
[[872 146]
 [134 955]]


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
PATCH_DIR = "/content/patches"
RESULTS_PATH = "/content/results.json"
CLASS_NAMES = ["non_cropland", "cropland"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 15
LR = 1e-3
IN_CHANNELS = 13  # 10 S2 bands + NDVI + NDWI + EVI

print(f"Using device: {DEVICE}")
if DEVICE.type == "cpu":
    print("No GPU detected — training will run on CPU and be slower. "
          "In Colab: Runtime > Change runtime type > GPU (T4), then re-run this cell.")


class PatchDataset(Dataset):
    def __init__(self, split):
        self.paths = sorted(glob.glob(os.path.join(PATCH_DIR, split, "*.npz")))
        if len(self.paths) == 0:
            raise RuntimeError(f"No patches found for split='{split}' in {PATCH_DIR}. "
                                f"Run Stage 3 first.")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        d = np.load(self.paths[idx])
        feats = d["features"].astype(np.float32)  # [13, H, W]
        # Per-sample normalization (mean/std) — simple and robust without
        # needing a separately-computed training-set-wide statistic.
        mean = feats.mean(axis=(1, 2), keepdims=True)
        std = feats.std(axis=(1, 2), keepdims=True) + 1e-6
        feats = (feats - mean) / std

        pixel_mask = d["pixel_mask"].astype(np.float32)  # [H, W], 0/1
        patch_label = int(d["label"])
        return torch.from_numpy(feats), torch.from_numpy(pixel_mask), patch_label


def build_model():
    model = smp.Unet(
        encoder_name="resnet18",
        encoder_weights=None,  # ImageNet weights don't apply to 13-channel input
        in_channels=IN_CHANNELS,
        classes=1,  # binary segmentation, single-channel logit
    )
    return model.to(DEVICE)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for feats, pixel_mask, _ in loader:
        feats, pixel_mask = feats.to(DEVICE), pixel_mask.to(DEVICE)
        optimizer.zero_grad()
        logits = model(feats).squeeze(1)  # [B, H, W]
        loss = criterion(logits, pixel_mask)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * feats.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_patch_preds, all_patch_labels = [], []
    for feats, pixel_mask, patch_label in loader:
        feats = feats.to(DEVICE)
        logits = model(feats).squeeze(1)
        pixel_probs = torch.sigmoid(logits)
        pixel_preds = (pixel_probs > 0.5).float().cpu()
        # Aggregate to patch level: majority of predicted-cropland pixels
        patch_pred = (pixel_preds.mean(dim=(1, 2)) > 0.5).long().numpy()
        all_patch_preds.extend(patch_pred.tolist())
        all_patch_labels.extend(patch_label.numpy().tolist())
    return np.array(all_patch_labels), np.array(all_patch_preds)


def append_results(entry):
    results = []
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH) as f:
            results = json.load(f)
    results.append(entry)
    with open(RESULTS_PATH, "w") as f:
        json.dump(results, f, indent=2)


def main():
    train_ds = PatchDataset("train")
    val_ds = PatchDataset("val")
    test_ds = PatchDataset("test")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = build_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCEWithLogitsLoss()

    mlflow.set_experiment("crop_classification")
    with mlflow.start_run(run_name="unet_segmentation"):
        mlflow.log_params(dict(
            encoder="resnet18", in_channels=IN_CHANNELS, epochs=EPOCHS,
            batch_size=BATCH_SIZE, lr=LR, device=str(DEVICE),
        ))

        for epoch in range(1, EPOCHS + 1):
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
            val_labels, val_preds = evaluate(model, val_loader)
            val_acc = accuracy_score(val_labels, val_preds) if len(val_labels) else float("nan")
            print(f"Epoch {epoch}/{EPOCHS}  train_loss={train_loss:.4f}  val_acc={val_acc:.4f}")
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_accuracy", val_acc, step=epoch)

        test_labels, test_preds = evaluate(model, test_loader)
        acc = accuracy_score(test_labels, test_preds)
        f1_macro = f1_score(test_labels, test_preds, average="macro")
        f1_per_class = f1_score(test_labels, test_preds, average=None, labels=[0, 1])
        cm = confusion_matrix(test_labels, test_preds, labels=[0, 1])

        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_f1_macro", f1_macro)
        for cname, f1c in zip(CLASS_NAMES, f1_per_class):
            mlflow.log_metric(f"test_f1_{cname}", f1c)

        print("\n=== UNet — Test Set (patch-level, aggregated from pixel predictions) ===")
        print(f"Accuracy: {acc:.4f}")
        print(f"Macro F1: {f1_macro:.4f}")
        print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES, labels=[0, 1]))
        print("Confusion matrix (rows=true, cols=pred):")
        print(cm)

        append_results({
            "model": "unet",
            "test_accuracy": float(acc),
            "test_f1_macro": float(f1_macro),
            "test_f1_per_class": {c: float(v) for c, v in zip(CLASS_NAMES, f1_per_class)},
            "confusion_matrix": cm.tolist(),
            "n_train": len(train_ds), "n_val": len(val_ds), "n_test": len(test_ds),
        })

    return model


if __name__ == "__main__":
    main()

Using device: cuda
Epoch 1/15  train_loss=0.5103  val_acc=0.8371
Epoch 2/15  train_loss=0.4638  val_acc=0.8542
Epoch 3/15  train_loss=0.4486  val_acc=0.8142
Epoch 4/15  train_loss=0.4328  val_acc=0.8428
Epoch 5/15  train_loss=0.4187  val_acc=0.7966
Epoch 6/15  train_loss=0.4115  val_acc=0.7987
Epoch 7/15  train_loss=0.4058  val_acc=0.8656
Epoch 8/15  train_loss=0.3934  val_acc=0.8609
Epoch 9/15  train_loss=0.3927  val_acc=0.7929
Epoch 10/15  train_loss=0.3832  val_acc=0.8573
Epoch 11/15  train_loss=0.3686  val_acc=0.8495
Epoch 12/15  train_loss=0.3640  val_acc=0.8422
Epoch 13/15  train_loss=0.3552  val_acc=0.8293
Epoch 14/15  train_loss=0.3455  val_acc=0.8293
Epoch 15/15  train_loss=0.3382  val_acc=0.8791

=== UNet — Test Set (patch-level, aggregated from pixel predictions) ===
Accuracy: 0.8771
Macro F1: 0.8768
              precision    recall  f1-score   support

non_cropland       0.88      0.86      0.87      1018
    cropland       0.87      0.89      0.88      1089

    accuracy 

In [15]:
# STAGE 4c  GEOSPATIAL FOUNDATION MODEL, FINE-TUNED VIA TORCHGEO

PATCH_DIR = "/content/patches"
RESULTS_PATH = "/content/results.json"
CLASS_NAMES = ["non_cropland", "cropland"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16
EPOCHS = 10
LR = 1e-4  # smaller LR — fine-tuning a pretrained encoder, not training from scratch
FREEZE_ENCODER_EPOCHS = 2  # linear-probe warmup before unfreezing, for stability

print(f"Using device: {DEVICE}")
if DEVICE.type == "cpu":
    print("No GPU detected — ViT fine-tuning will be slow on CPU. "
          "In Colab: Runtime > Change runtime type > GPU (T4).")

# Our exported band order (Stage 1 EXPORT_BANDS) -> position in the
# 13-band order the pretrained model expects.
OUR_BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
TARGET_BANDS = ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8a", "B9", "B10", "B11", "B12"]
# map our band name -> target band name (B8A vs B8a casing)
NAME_FIX = {"B8A": "B8a"}
OUR_TO_TARGET_IDX = [TARGET_BANDS.index(NAME_FIX.get(b, b)) for b in OUR_BANDS]


def remap_to_13_band(patch_10band):
    """patch_10band: [10, H, W] -> zero-padded [13, H, W] in TARGET_BANDS order."""
    c, h, w = patch_10band.shape
    out = np.zeros((13, h, w), dtype=np.float32)
    for src_idx, tgt_idx in enumerate(OUR_TO_TARGET_IDX):
        out[tgt_idx] = patch_10band[src_idx]
    return out


class PatchDataset(Dataset):
    def __init__(self, split):
        self.paths = sorted(glob.glob(os.path.join(PATCH_DIR, split, "*.npz")))
        if len(self.paths) == 0:
            raise RuntimeError(f"No patches found for split='{split}' in {PATCH_DIR}. "
                                f"Run Stage 3 first.")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        d = np.load(self.paths[idx])
        feats = d["features"].astype(np.float32)[:10]  # drop the 3 index channels, raw bands only
        feats = remap_to_13_band(feats)
        mean = feats.mean(axis=(1, 2), keepdims=True)
        std = feats.std(axis=(1, 2), keepdims=True) + 1e-6
        feats = (feats - mean) / std
        label = int(d["label"])
        return torch.from_numpy(feats), label


def build_model():
    weights = ViTSmall16_Weights.SENTINEL2_ALL_MAE
    model = timm.create_model("vit_small_patch16_224", in_chans=13, num_classes=2)
    try:
        state_dict = weights.get_state_dict(progress=True)
        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        print(f"Loaded pretrained SSL4EO-S12 MAE weights. "
              f"({len(missing)} params randomly init'd, e.g. classifier head)")
    except Exception as e:
        print(f"WARNING: could not download pretrained weights ({e}). "
              f"Falling back to random init — check internet access in this Colab session.")
    return model.to(DEVICE)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for feats, labels in loader:
        feats = F.interpolate(feats, size=(224, 224), mode="bilinear", align_corners=False)
        feats, labels = feats.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(feats)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * feats.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for feats, labels in loader:
        feats = F.interpolate(feats, size=(224, 224), mode="bilinear", align_corners=False)
        feats = feats.to(DEVICE)
        logits = model(feats)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())
    return np.array(all_labels), np.array(all_preds)


def set_encoder_trainable(model, trainable):
    for name, param in model.named_parameters():
        if "head" not in name:  # timm ViT classifier head is named "head"
            param.requires_grad = trainable


def append_results(entry):
    results = []
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH) as f:
            results = json.load(f)
    results.append(entry)
    with open(RESULTS_PATH, "w") as f:
        json.dump(results, f, indent=2)


def main():
    train_ds = PatchDataset("train")
    val_ds = PatchDataset("val")
    test_ds = PatchDataset("test")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = build_model()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    mlflow.set_experiment("crop_classification")
    with mlflow.start_run(run_name="foundation_model_vit_ssl4eo_s12_mae"):
        mlflow.log_params(dict(
            base_model="ViTSmall16_Weights.SENTINEL2_ALL_MAE (SatMAE substitute)",
            epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=str(DEVICE),
            freeze_encoder_epochs=FREEZE_ENCODER_EPOCHS,
        ))

        for epoch in range(1, EPOCHS + 1):
            set_encoder_trainable(model, trainable=(epoch > FREEZE_ENCODER_EPOCHS))
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
            val_labels, val_preds = evaluate(model, val_loader)
            val_acc = accuracy_score(val_labels, val_preds) if len(val_labels) else float("nan")
            phase = "linear-probe" if epoch <= FREEZE_ENCODER_EPOCHS else "fine-tune"
            print(f"Epoch {epoch}/{EPOCHS} [{phase}]  train_loss={train_loss:.4f}  val_acc={val_acc:.4f}")
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_accuracy", val_acc, step=epoch)

        test_labels, test_preds = evaluate(model, test_loader)
        acc = accuracy_score(test_labels, test_preds)
        f1_macro = f1_score(test_labels, test_preds, average="macro")
        f1_per_class = f1_score(test_labels, test_preds, average=None, labels=[0, 1])
        cm = confusion_matrix(test_labels, test_preds, labels=[0, 1])

        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_f1_macro", f1_macro)
        for cname, f1c in zip(CLASS_NAMES, f1_per_class):
            mlflow.log_metric(f"test_f1_{cname}", f1c)

        print("\n=== Foundation Model (ViT / SSL4EO-S12 MAE) — Test Set ===")
        print(f"Accuracy: {acc:.4f}")
        print(f"Macro F1: {f1_macro:.4f}")
        print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES, labels=[0, 1]))
        print("Confusion matrix (rows=true, cols=pred):")
        print(cm)

        append_results({
            "model": "foundation_model_vit_ssl4eo_s12_mae",
            "test_accuracy": float(acc),
            "test_f1_macro": float(f1_macro),
            "test_f1_per_class": {c: float(v) for c, v in zip(CLASS_NAMES, f1_per_class)},
            "confusion_matrix": cm.tolist(),
            "n_train": len(train_ds), "n_val": len(val_ds), "n_test": len(test_ds),
        })

    return model


if __name__ == "__main__":
    main()

Using device: cuda
Downloading: "https://huggingface.co/wangyi111/SSL4EO-S12/resolve/75c72195d35201dc1fb210818993518c25da566b/B13_vits16_mae_ep99_enc.pth" to /root/.cache/torch/hub/checkpoints/B13_vits16_mae_ep99_enc.pth


100%|██████████| 86.5M/86.5M [00:10<00:00, 8.45MB/s]


Loaded pretrained SSL4EO-S12 MAE weights. (2 params randomly init'd, e.g. classifier head)
Epoch 1/10 [linear-probe]  train_loss=0.6041  val_acc=0.7094
Epoch 2/10 [linear-probe]  train_loss=0.5325  val_acc=0.7307
Epoch 3/10 [fine-tune]  train_loss=0.4651  val_acc=0.8350
Epoch 4/10 [fine-tune]  train_loss=0.3506  val_acc=0.8329
Epoch 5/10 [fine-tune]  train_loss=0.3054  val_acc=0.8158
Epoch 6/10 [fine-tune]  train_loss=0.2668  val_acc=0.8075
Epoch 7/10 [fine-tune]  train_loss=0.2430  val_acc=0.8588
Epoch 8/10 [fine-tune]  train_loss=0.2104  val_acc=0.8729
Epoch 9/10 [fine-tune]  train_loss=0.1881  val_acc=0.8308
Epoch 10/10 [fine-tune]  train_loss=0.1672  val_acc=0.8547

=== Foundation Model (ViT / SSL4EO-S12 MAE) — Test Set ===
Accuracy: 0.8714
Macro F1: 0.8713
              precision    recall  f1-score   support

non_cropland       0.86      0.88      0.87      1018
    cropland       0.89      0.86      0.87      1089

    accuracy                           0.87      2107
   macro a

In [17]:
# RESULTS COMPARISON
RESULTS_PATH = "/content/results.json"
OUTPUT_DIR = "/content/deliverables"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_DISPLAY_NAMES = {
    "random_forest": "Random Forest\n(hand-crafted features)",
    "unet": "UNet\n(ResNet18 encoder)",
    "foundation_model_vit_ssl4eo_s12_mae": "ViT Foundation Model\n(SSL4EO-S12 MAE, SatMAE substitute)",
}


def load_results():
    if not os.path.exists(RESULTS_PATH):
        raise RuntimeError(
            f"{RESULTS_PATH} not found. Run Stages 4a/4b/4c first — each "
            f"appends its results to this file."
        )
    with open(RESULTS_PATH) as f:
        results = json.load(f)

    # keep only the LATEST run per model (in case a stage was re-run)
    latest = {}
    for r in results:
        latest[r["model"]] = r
    return list(latest.values())


def build_table(results):
    rows = []
    for r in results:
        row = {
            "Model": MODEL_DISPLAY_NAMES.get(r["model"], r["model"]).replace("\n", " "),
            "Accuracy": round(r["test_accuracy"], 4),
            "Macro F1": round(r["test_f1_macro"], 4),
            "F1 (non-cropland)": round(r["test_f1_per_class"].get("non_cropland", float("nan")), 4),
            "F1 (cropland)": round(r["test_f1_per_class"].get("cropland", float("nan")), 4),
            "N test patches": r.get("n_test", "-"),
        }
        rows.append(row)
    df = pd.DataFrame(rows).sort_values("Macro F1", ascending=False).reset_index(drop=True)
    return df


def plot_comparison(results, out_path):
    models = [MODEL_DISPLAY_NAMES.get(r["model"], r["model"]) for r in results]
    acc = [r["test_accuracy"] for r in results]
    f1 = [r["test_f1_macro"] for r in results]

    x = range(len(models))
    width = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar([i - width / 2 for i in x], acc, width, label="Accuracy")
    ax.bar([i + width / 2 for i in x], f1, width, label="Macro F1")
    ax.set_xticks(list(x))
    ax.set_xticklabels(models, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score")
    ax.set_title("Crop (Cropland) Classification — Model Comparison, Test Set")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def main():
    results = load_results()
    present = {r["model"] for r in results}
    expected = set(MODEL_DISPLAY_NAMES.keys())
    missing = expected - present
    if missing:
        print(f"WARNING: missing results for: {sorted(missing)} — "
              f"run those Stage 4 scripts before treating this as final.")

    df = build_table(results)
    print("\n=== Model Comparison ===")
    print(df.to_string(index=False))

    csv_path = os.path.join(OUTPUT_DIR, "results_comparison_table.csv")
    df.to_csv(csv_path, index=False)
    print(f"\nSaved table: {csv_path}")

    chart_path = os.path.join(OUTPUT_DIR, "results_comparison_chart.png")
    plot_comparison(results, chart_path)
    print(f"Saved chart: {chart_path}")

    return df


if __name__ == "__main__":
    main()


=== Model Comparison ===
                                                   Model  Accuracy  Macro F1  F1 (non-cropland)  F1 (cropland)  N test patches
                                 UNet (ResNet18 encoder)    0.8771    0.8768             0.8711         0.8825            2107
ViT Foundation Model (SSL4EO-S12 MAE, SatMAE substitute)    0.8714    0.8713             0.8689         0.8738            2107
                   Random Forest (hand-crafted features)    0.8671    0.8669             0.8617         0.8721            2107

Saved table: /content/deliverables/results_comparison_table.csv
Saved chart: /content/deliverables/results_comparison_chart.png
